# Tutorial 2: Gene (DNA) Embeddings

This notebook shows how to compute **DNA-level gene embeddings** with `embpy`.

Available DNA embedding models:

| Model key | Architecture | Notes |
|---|---|---|
| `enformer_human_rough` | Enformer | 896 output bins × 5 313 tracks → pooled |
| `borzoi_v0` / `borzoi_v1` | Borzoi | Similar genomic track prediction |
| `evo2_7b` / `evo2_40b` | Evo2 | DNA language model (requires GPU) |
| `evo2_7b_base` / `evo2_1b_base` | Evo2 (base) | Smaller checkpoints |

All models accept raw DNA sequences and return dense embeddings.
The `BioEmbedder` class handles sequence resolution and model dispatch
transparently.

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

In [2]:
from embpy.embedder import BioEmbedder

# BioEmbedder auto-detects the best device (GPU if available)
embedder = BioEmbedder(device="auto")
print(f"Device: {embedder.device}")
print(f"Available models: {embedder.list_available_models()}")

Device: cuda
Available models: ['atom_pair_count_fp', 'atom_pair_fp', 'bert_base_uncased', 'borzoi_v0', 'borzoi_v1', 'caduceus_ph_131k', 'caduceus_ps_131k', 'chemberta2MLM', 'chemberta2MTR', 'enformer_human_rough', 'esm2_150M', 'esm2_35M', 'esm2_3B', 'esm2_650M', 'esm2_8M', 'esmc_300m', 'esmc_600m', 'evo1.5_8k', 'evo1_131k', 'evo1_8k', 'evo1_crispr', 'evo1_transposon', 'evo2_1b_base', 'evo2_40b', 'evo2_7b', 'evo2_7b_base', 'gena_lm_bert_base', 'gena_lm_bert_base_multi', 'gena_lm_bert_large', 'gena_lm_bigbird_base', 'hyenadna_large_1m', 'hyenadna_medium_160k', 'hyenadna_medium_450k', 'hyenadna_small_32k', 'hyenadna_tiny_1k', 'maccs_fp', 'mhg_gnn', 'minilm_l6_v2', 'minimol', 'mole', 'molformer_base', 'morgan_count_fp', 'morgan_fp', 'nt_2b5_1000g', 'nt_2b5_multi', 'nt_500m_1000g', 'nt_500m_human_ref', 'nt_v2_100m', 'nt_v2_250m', 'nt_v2_500m', 'nt_v2_50m', 'ntv3_100m_pos', 'ntv3_100m_pre', 'ntv3_650m_pos', 'ntv3_650m_pre', 'ntv3_8m_pre', 'prot_t5_xl', 'prot_t5_xl_half', 'rdkit_fp', 'torsio

## 1. Embed a single gene by symbol

Provide a human gene symbol and let `embpy` resolve the DNA sequence via
the Ensembl REST API, then embed it.

In [3]:
import numpy as np

# Choose a model (use enformer_human_rough as the default example)
MODEL = "enformer_human_rough"

emb = embedder.embed_gene(
    identifier="TP53",
    model=MODEL,
    id_type="symbol",
    organism="human",
    pooling_strategy="mean",
)

print(f"Embedding shape: {emb.shape}")
print(f"Embedding dtype: {emb.dtype}")
print(f"First 10 values: {emb[:10]}")

Embedding shape: (3072,)
Embedding dtype: float32
First 10 values: [-1.3462065e-01 -5.3989777e-05  4.3073799e-02 -6.9053367e-02
 -1.5665933e-01  9.0005778e-02 -1.2748341e-01 -6.0363330e-02
 -1.0527112e-01  5.2237257e-02]


## 2. Embed a gene by Ensembl ID

In [4]:
emb_ens = embedder.embed_gene(
    identifier="ENSG00000141510",   # TP53
    model=MODEL,
    id_type="ensembl_id",
    pooling_strategy="mean",
)
print(f"Embedding shape: {emb_ens.shape}")

Embedding shape: (3072,)


## 3. Embed a raw DNA sequence directly

If you already have a DNA sequence, you can pass it as the identifier
and the embedder will detect it automatically.

In [5]:
# A synthetic DNA sequence (must be long enough for Enformer: 196_608 bp)
# For demonstration, we generate a random sequence.
rng = np.random.default_rng(42)
dna = "".join(rng.choice(list("ACGT"), size=196_608))

emb_raw = embedder.embed_gene(
    identifier=dna,
    model=MODEL,
    id_type="sequence",
    pooling_strategy="mean",
)
print(f"Raw DNA embedding shape: {emb_raw.shape}")

Raw DNA embedding shape: (3072,)


## 3b. Embed only exons or introns

You can embed specific gene regions — only exonic or intronic sequences —
using the `region` parameter. This uses the Ensembl REST API to fetch
exon coordinates from the canonical transcript and extracts each region's
DNA sequence.

Available values: `"full"` (default, entire gene), `"exons"`, `"introns"`.

In [ ]:
# Inspect the gene structure first
from embpy.resources.gene_resolver import GeneResolver

resolver = GeneResolver()
exons = resolver.get_gene_regions("TP53", region="exons")
introns = resolver.get_gene_regions("TP53", region="introns")

if exons:
    print(f"TP53 exons: {len(exons)} regions")
    for ex in exons[:3]:
        print(f"  {ex['id']}: {ex['start']}-{ex['end']} ({len(str(ex['sequence']))} bp)")
if introns:
    print(f"\nTP53 introns: {len(introns)} regions")
    for intron in introns[:3]:
        print(f"  {intron['id']}: {intron['start']}-{intron['end']} ({len(str(intron['sequence']))} bp)")

In [ ]:
# Embed only exons
emb_exons = embedder.embed_gene(
    identifier="TP53",
    model=MODEL,
    region="exons",
    pooling_strategy="mean",
)
print(f"TP53 exon-only embedding: shape={emb_exons.shape}")

# Embed only introns
emb_introns = embedder.embed_gene(
    identifier="TP53",
    model=MODEL,
    region="introns",
    pooling_strategy="mean",
)
print(f"TP53 intron-only embedding: shape={emb_introns.shape}")

# Compare with full gene embedding
from numpy.linalg import norm

cos_exon_full = np.dot(emb, emb_exons) / (norm(emb) * norm(emb_exons))
cos_intron_full = np.dot(emb, emb_introns) / (norm(emb) * norm(emb_introns))
cos_exon_intron = np.dot(emb_exons, emb_introns) / (norm(emb_exons) * norm(emb_introns))

print(f"\nCosine similarities:")
print(f"  full vs exons:   {cos_exon_full:.4f}")
print(f"  full vs introns: {cos_intron_full:.4f}")
print(f"  exons vs introns: {cos_exon_intron:.4f}")

## 4. Batch embedding

Use `embed_genes_batch` to embed multiple genes efficiently. Failed
lookups return `None` without crashing the batch.

In [6]:
genes = ["TP53", "BRCA1", "EGFR", "MYC"]

embeddings = embedder.embed_genes_batch(
    model=MODEL,
    identifiers=genes,
    id_type="symbol",
    organism="human",
    pooling_strategy="mean",
)

for gene, emb in zip(genes, embeddings):
    shape = emb.shape if emb is not None else "FAILED"
    print(f"  {gene:10s} → {shape}")

  TP53       → (3072,)
  BRCA1      → (3072,)
  EGFR       → (3072,)
  MYC        → (3072,)


## 5. Pooling strategies

DNA models often produce multi-dimensional outputs (e.g. Enformer outputs
896 spatial bins × 5 313 tracks). The `pooling_strategy` parameter controls
how these are reduced to a single vector:

- `"mean"` – average over the spatial / sequence dimension
- `"max"` – max-pool over the spatial / sequence dimension
- `"cls"` – take the first token (for transformer models)

In [7]:
for strategy in ["mean", "max"]:
    emb = embedder.embed_gene(
        "TP53", model=MODEL, pooling_strategy=strategy,
    )
    print(f"  pooling={strategy:5s} → shape {emb.shape}, mean={emb.mean():.4f}")

  pooling=mean  → shape (3072,), mean=-0.0040
  pooling=max   → shape (3072,), mean=0.6709


## 6. Using Evo2 (DNA language model)

Evo2 is a large DNA language model that requires a GPU. If a CUDA device
is available, you can use it like any other model.

In [8]:
import gc
import torch

if torch.cuda.is_available():
    # Evo2 7B needs ~15 GB VRAM — free other models first
    embedder.model_cache.clear()
    gc.collect()
    torch.cuda.empty_cache()

    try:
        emb_evo2 = embedder.embed_gene(
            identifier="TP53",
            model="evo2_7b",
            id_type="symbol",
            pooling_strategy="mean",
        )
        print(f"Evo2 embedding shape: {emb_evo2.shape}")
    except RuntimeError as e:
        print(f"Evo2 failed: {e}")
else:
    print("No CUDA device found – skipping Evo2 (requires GPU).")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Found complete file in repo: evo2_7b.pt


/ictstr01/home/icb/goncalo.pinto/tools/apps/mamba/envs/embpy/lib/python3.12/site-packages/evo2/models.py:282: UserWarning: Transformer Engine not installed. Falling back to bf16 projections (use_fp8_input_projections=False). 
  warnings.warn(
100%|██████████| 32/32 [00:00<00:00, 89.35it/s]


Extra keys in state_dict: {'blocks.11.projections._extra_state', 'blocks.17.mixer.attn._extra_state', 'blocks.22.projections._extra_state', 'blocks.13.mixer.mixer.filter.t', 'blocks.6.projections._extra_state', 'blocks.19.projections._extra_state', 'blocks.20.projections._extra_state', 'blocks.2.projections._extra_state', 'blocks.2.mixer.mixer.filter.t', 'blocks.4.projections._extra_state', 'blocks.12.projections._extra_state', 'blocks.10.mixer.dense._extra_state', 'blocks.25.projections._extra_state', 'unembed.weight', 'blocks.27.projections._extra_state', 'blocks.1.projections._extra_state', 'blocks.5.projections._extra_state', 'blocks.23.projections._extra_state', 'blocks.16.mixer.mixer.filter.t', 'blocks.26.projections._extra_state', 'blocks.31.mixer.attn._extra_state', 'blocks.20.mixer.mixer.filter.t', 'blocks.6.mixer.mixer.filter.t', 'blocks.24.mixer.dense._extra_state', 'blocks.9.projections._extra_state', 'blocks.29.projections._extra_state', 'blocks.21.projections._extra_state

ERROR:root:Embedding error for TP53 with model evo2_7b: schema_.has_value() INTERNAL ASSERT FAILED at "/pytorch/aten/src/ATen/core/dispatch/OperatorEntry.h":80, please report a bug to PyTorch. Tried to access the schema for  which doesn't have a schema registered yet


Evo2 failed: Embedding failed for TP53


## 7. Comparing embeddings across models

Embeddings from different models live in different-dimensional spaces.
Later, in [Tutorial 6](06_combined_analysis.ipynb), we will show how to
align them via PCA.

## Downloading the TRADE / Nadig et al. 2025 Datasets

The [TRADE paper](https://www.nature.com/articles/s41588-025-02169-3) (Nadig et al., *Nat Genet* 2025)
analyzed several Perturb-seq datasets beyond the original Replogle 2022 screens.
We download three additional datasets:

| Dataset | Cell type | Source | Format |
|---|---|---|---|
| Jurkat-Essential (CRISPRi) | Jurkat | [scPerturb / Zenodo](https://zenodo.org/records/13350497) | h5ad |
| HepG2-Essential (CRISPRi) | HepG2 | [scPerturb / Zenodo](https://zenodo.org/records/13350497) | h5ad |
| K562-Titration (CRISPRi) | K562 | [GEO GSE132080](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE132080) (Jost et al. 2020) | MTX + metadata |

In [ ]:
import subprocess, os

NADIG_DIR = DATA_DIR / "nadig"
JOST_DIR = DATA_DIR / "jost"
NADIG_DIR.mkdir(parents=True, exist_ok=True)
JOST_DIR.mkdir(parents=True, exist_ok=True)

SCPERTURB_BASE = "https://zenodo.org/api/records/13350497/files"
GEO_BASE = "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE132080&format=file&file="

downloads = [
    # Nadig / TRADE -- Jurkat & HepG2 (harmonized h5ad from scPerturb)
    (f"{SCPERTURB_BASE}/NadigOConner2024_jurkat.h5ad/content",
     NADIG_DIR / "NadigOConner2024_jurkat.h5ad"),
    (f"{SCPERTURB_BASE}/NadigOConner2024_hepg2.h5ad/content",
     NADIG_DIR / "NadigOConner2024_hepg2.h5ad"),
    # Jost 2020 -- K562 Titration (raw from GEO GSE132080)
    (f"{GEO_BASE}GSE132080%5F10X%5Fmatrix%2Emtx%2Egz",
     JOST_DIR / "GSE132080_10X_matrix.mtx.gz"),
    (f"{GEO_BASE}GSE132080%5F10X%5Fbarcodes%2Etsv%2Egz",
     JOST_DIR / "GSE132080_10X_barcodes.tsv.gz"),
    (f"{GEO_BASE}GSE132080%5F10X%5Fgenes%2Etsv%2Egz",
     JOST_DIR / "GSE132080_10X_genes.tsv.gz"),
    (f"{GEO_BASE}GSE132080%5Fcell%5Fidentities%2Ecsv%2Egz",
     JOST_DIR / "GSE132080_cell_identities.csv.gz"),
    (f"{GEO_BASE}GSE132080%5FsgRNA%5Fbarcode%5Fsequences%5Fand%5Fphenotypes%2Ecsv%2Egz",
     JOST_DIR / "GSE132080_sgRNA_barcode_sequences_and_phenotypes.csv.gz"),
]

for url, dest in downloads:
    if dest.exists():
        print(f"Already exists: {dest.name}")
        continue
    print(f"Downloading {dest.name} ...")
    subprocess.run(
        ["wget", "-q", "--show-progress", "-O", str(dest), url],
        check=True,
    )
    print(f"  -> {dest.stat().st_size / 1e9:.2f} GB")

print("\nAll downloads complete.")

In [ ]:
import scanpy as sc
import pandas as pd

# --- Nadig 2024: Jurkat-Essential ---
nadig_jurkat = ad.read_h5ad(NADIG_DIR / "NadigOConner2024_jurkat.h5ad")
print(f"Nadig Jurkat: {nadig_jurkat.shape[0]:,} cells x {nadig_jurkat.shape[1]:,} genes")
print(nadig_jurkat)

# --- Nadig 2024: HepG2-Essential ---
nadig_hepg2 = ad.read_h5ad(NADIG_DIR / "NadigOConner2024_hepg2.h5ad")
print(f"\nNadig HepG2: {nadig_hepg2.shape[0]:,} cells x {nadig_hepg2.shape[1]:,} genes")
print(nadig_hepg2)

# --- Jost 2020: K562-Titration ---
jost = sc.read_mtx(JOST_DIR / "GSE132080_10X_matrix.mtx.gz").T

barcodes = pd.read_csv(
    JOST_DIR / "GSE132080_10X_barcodes.tsv.gz", header=None, sep="\t"
)
jost.obs_names = barcodes[0].values

genes = pd.read_csv(
    JOST_DIR / "GSE132080_10X_genes.tsv.gz", header=None, sep="\t"
)
jost.var_names = genes[1].values
jost.var["gene_id"] = genes[0].values

cell_ids = pd.read_csv(JOST_DIR / "GSE132080_cell_identities.csv.gz", index_col=0)
jost.obs = jost.obs.join(cell_ids, how="left")

sgrna_info = pd.read_csv(
    JOST_DIR / "GSE132080_sgRNA_barcode_sequences_and_phenotypes.csv.gz"
)

print(f"\nJost K562 Titration: {jost.shape[0]:,} cells x {jost.shape[1]:,} genes")
print(jost)
print(f"\nsgRNA phenotype table: {sgrna_info.shape}")
print(sgrna_info.head())

In [9]:
# Quick summary of dimension per model
for m in ["enformer_human_rough", "borzoi_v0"]:
    try:
        e = embedder.embed_gene("TP53", model=m, pooling_strategy="mean")
        print(f"  {m:30s} → dim {e.shape[0]:,}")
    except Exception as exc:
        print(f"  {m:30s} → {exc}")

  enformer_human_rough           → dim 3,072
  borzoi_v0                      → dim 1,536


# Loading the Perturb-Seq Datasets

In [10]:
import anndata as ad
from pathlib import Path

DATA_DIR = Path("/lustre/groups/ml01/workspace/goncalo.pinto/embpy/data")

# Norman 2019
norman = ad.read_h5ad(DATA_DIR / "norman" / "norman_2019.h5ad")
print(f"Norman 2019: {norman.shape[0]:,} cells x {norman.shape[1]:,} genes")
print(norman)

# Replogle 2022 - three subsets
replogle_essential = ad.read_h5ad(DATA_DIR / "replogle" / "replogle_2022_k562_essential.h5ad")
print(f"\nReplogle K562 essential: {replogle_essential.shape[0]:,} cells x {replogle_essential.shape[1]:,} genes")
print(replogle_essential)

replogle_gwps = ad.read_h5ad(DATA_DIR / "replogle" / "replogle_2022_k562_gwps.h5ad")
print(f"\nReplogle K562 GWPS: {replogle_gwps.shape[0]:,} cells x {replogle_gwps.shape[1]:,} genes")
print(replogle_gwps)

replogle_rpe1 = ad.read_h5ad(DATA_DIR / "replogle" / "replogle_2022_rpe1.h5ad")
print(f"\nReplogle RPE1: {replogle_rpe1.shape[0]:,} cells x {replogle_rpe1.shape[1]:,} genes")
print(replogle_rpe1)

Norman 2019: 111,255 cells x 19,018 genes
AnnData object with n_obs × n_vars = 111255 × 19018
    obs: 'guide_identity', 'read_count', 'UMI_count', 'coverage', 'gemgroup', 'good_coverage', 'number_of_cells', 'guide_AHR', 'guide_ARID1A', 'guide_ARRDC3', 'guide_ATL1', 'guide_BAK1', 'guide_BCL2L11', 'guide_BCORL1', 'guide_BPGM', 'guide_C19orf26', 'guide_C3orf72', 'guide_CBFA2T3', 'guide_CBL', 'guide_CDKN1A', 'guide_CDKN1B', 'guide_CDKN1C', 'guide_CEBPA', 'guide_CEBPB', 'guide_CEBPE', 'guide_CELF2', 'guide_CITED1', 'guide_CKS1B', 'guide_CLDN6', 'guide_CNN1', 'guide_CNNM4', 'guide_COL1A1', 'guide_COL2A1', 'guide_CSRNP1', 'guide_DLX2', 'guide_DUSP9', 'guide_EGR1', 'guide_ELMSAN1', 'guide_ETS2', 'guide_FEV', 'guide_FOSB', 'guide_FOXA1', 'guide_FOXA3', 'guide_FOXF1', 'guide_FOXL2', 'guide_FOXO4', 'guide_GLB1L2', 'guide_HES7', 'guide_HK2', 'guide_HNF4A', 'guide_HOXA13', 'guide_HOXB9', 'guide_HOXC13', 'guide_IER5L', 'guide_IGDCC3', 'guide_IKZF3', 'guide_IRF1', 'guide_ISL2', 'guide_JUN', 'guide_K

## Summary

| Method | Use case |
|---|---|
| `embed_gene(identifier, model)` | Single gene (symbol, Ensembl ID, or raw DNA) |
| `embed_gene(..., id_type="sequence")` | Raw DNA sequence directly |
| `embed_gene(..., region="exons")` | Embed only exonic regions |
| `embed_gene(..., region="introns")` | Embed only intronic regions |
| `embed_genes_batch(model, identifiers)` | Batch of genes |
| `pooling_strategy` kwarg | Control how spatial dims are reduced |
| `GeneResolver.get_gene_regions(gene, region=...)` | Extract exon/intron sequences |

Next: [Tutorial 3 – Protein Embeddings](03_protein_embeddings.ipynb)